In [ ]:
# Import libraries
from pyspark.sql import SparkSession
from lib.helpers import get_run_date
from lib.job_control import get_max_insert_ts, insert_log
from pyspark.sql.functions import current_timestamp, col, lit, to_date, date_sub, max as spark_max, row_number, coalesce
from pyspark.sql.window import Window
from lib.aws_s3 import warehouse_dir
from delta.tables import DeltaTable

In [ ]:
# Create the SparkSession
"""
spark = (
    SparkSession.builder
    .appName("05 - Dealer Production")
    .config("spark.sql.warehouse.dir", warehouse_dir)
    .enableHiveSupport()
    .getOrCreate()
)

spark
"""

In [ ]:
# Job parameters
now_ts = spark.range(1).select(current_timestamp().alias("now")).first()["now"]
run_date = get_run_date()
table = "dealer"
schema_name = "prd"
table_name = f"dim_{table}"
table_full_name = f"{schema_name}.{table_name}"
staging_table_full_name = f"stg.dim_{table}_stg"

print("****************************************")
print("05 - Dealer Production")
print("****************************************")
print(f"now_ts = {now_ts}")
print(f"run_date = {run_date}")
print(f"table = {table}")
print(f"schema_name = {schema_name}")
print(f"table_name = {table_name}")
print(f"table_full_name = {table_full_name}")
print(f"staging_table_full_name = {staging_table_full_name}")
print("")

In [ ]:
# Get max insert_ts for {schema_name}.{table_name} table from prd.job_control
max_insert_ts = get_max_insert_ts(spark, schema_name, table_name)
print(f"Max insert_ts in {table_full_name}: {max_insert_ts}\n")

In [ ]:
# Read data from staging
df_stg = (
    spark
    .table(staging_table_full_name)
)

In [ ]:
print(f"df_stg: {df_stg.count()} records read from {staging_table_full_name}")
df_stg.show()
df_stg.printSchema()

In [ ]:
df_dim_temp = (
    df_stg
    .withColumn("insert_ts", lit(now_ts))
    .withColumn("update_ts", lit(now_ts))
    .withColumn("hist_record_is_current", lit(False))
    .withColumn("hist_record_update_ts", lit(now_ts))
    .withColumn("hist_record_effective_end_date",date_sub(to_date(col("run_date"), "yyyyMMdd"), 1))
    .drop("run_date")
)

In [ ]:
print("Added columns for SCD update")
print(f"df_dim_temp: {df_dim_temp.count()} records")
df_dim_temp.show()
df_dim_temp.printSchema()

In [ ]:
print(f"{table_full_name}: Before upsert")
spark.table(f'{schema_name}.dim_{table}').show(truncate=False)

In [ ]:
# SCD1 & SCD2 update/insert to prd.dim_dealer
dt_dim = DeltaTable.forName(spark, table_full_name)

scd1_cols = ["dealer_name", "phone_number"]
scd2_cols = ["oem_name", "address", "city", "state", "zipcode"]

src = df_dim_temp  # latest incoming snapshot, one row per dealer_id

# Columns written on insert (dealer_key is added in row_b below)
new_cols = [c for c in src.columns if not c.startswith("hist_record_")] + ["dealer_key"]
key_type = src.schema["dealer_id"].dataType.simpleString()

# Current rows in the target for detecting SCD2 changes
cur = (
    spark.table(table_full_name)
    .where("is_current")
    .selectExpr("dealer_id", "TRUE AS cur_exists",
                *[f"{c} AS cur_{c}" for c in scd2_cols])
)

scd2_changed = " OR ".join(f"NOT ({c} <=> cur_{c})" for c in scd2_cols)

flagged = (
    src.join(cur, "dealer_id", "left")
       .selectExpr(
           "*",
           f"(cur_exists IS NOT NULL AND ({scd2_changed})) AS needs_close",
           f"(cur_exists IS NULL OR ({scd2_changed})) AS needs_insert",
       )
)

# Current highest key in the dimension (0 if the table is empty)
max_key = (
    spark.table(table_full_name)
    .agg(coalesce(spark_max("dealer_key"), lit(0)).alias("max_key"))
    .first()["max_key"]
)

# Windows to assign new keys to new dealers only
w = Window.orderBy("dealer_id")

# row_a: matches ALL existing versions of the dealer (SCD1 overwrite + closing).
# dealer_key is a null placeholder so the union lines up; it never reaches the target.
row_a = (
    flagged.where("cur_exists IS NOT NULL")
           .selectExpr("*", "dealer_id AS merge_key")
           .withColumn("dealer_key", lit(None).cast("long"))
)

# row_b: only the rows that insert (new dealers and new SCD2 versions).
# Keys are assigned here, so they stay contiguous.
row_b = (
    flagged.where("needs_insert")
           .selectExpr("*", f"CAST(NULL AS {key_type}) AS merge_key")
           .withColumn("dealer_key", (row_number().over(w) + lit(max_key)).cast("long"))
)

# Cache and materialize so both of Delta's passes see identical data
staged = row_a.unionByName(row_b).cache()
staged.count()

# Merge
(
    dt_dim.alias("t")
    .merge(staged.alias("s"), "t.dealer_id = s.merge_key")
    
    # 1) current row with an SCD2 change: close it (and apply SCD1 values)
    .whenMatchedUpdate(
        condition="t.is_current AND s.needs_close",
        set={
            "dealer_name": "s.dealer_name",
            "phone_number": "s.phone_number",
            "is_current": "s.hist_record_is_current",
            "update_ts": "s.hist_record_update_ts",
            "effective_end_date": "s.hist_record_effective_end_date",
        },
    )
    
    # 2) every other existing row: SCD1 overwrite if anything differs
    .whenMatchedUpdate(
        condition=" OR ".join(f"NOT (t.{c} <=> s.{c})" for c in scd1_cols),
        set={
            "dealer_name": "s.dealer_name",
            "phone_number": "s.phone_number",
            "update_ts": "s.hist_record_update_ts",
        },
    )
    
    # 3) new dealers and new SCD2 versions
    .whenNotMatchedInsert(
        condition="s.merge_key IS NULL",
        values={c: f"s.{c}" for c in new_cols},
    )
    .execute()
)

staged.unpersist()

In [ ]:
print(f"{table_full_name}: After upsert (Impacted records)")
spark.table(f'{schema_name}.dim_{table}').where(f"update_ts = '{now_ts}'").show()

In [ ]:
# Add entry to prd.job_control
if insert_log(spark, schema_name, table_name, now_ts, run_date):
    print(f"{schema_name}.{table_name} entry added to prd.job_control")

In [ ]:
# Review added entry
spark.sql(f"select * from prd.job_control where table_name = '{table_name}' order by insert_ts desc limit 1").show(truncate=False)

In [ ]:
# Stop spark session
#spark.stop()